# Generating text with Qwen3 0.6B

Chapter 2 of *Build a Reasoning Model (From Scratch)*. Loads the pretrained
Qwen3 0.6B **base** model and generates text locally — no API calls.

Run `python download_model.py` first if `qwen3/` is empty.

## 1. Setup

In [ ]:
import time
from pathlib import Path

import torch
from reasoning_from_scratch.ch02 import (
    generate_stats,
    generate_text_basic_stream,
    generate_text_basic_stream_cache,
    get_device,
)
from reasoning_from_scratch.qwen3 import (
    QWEN_CONFIG_06_B,
    Qwen3Model,
    Qwen3Tokenizer,
)

print(f"PyTorch version {torch.__version__}")
device = get_device()

# Uncomment to force CPU, as the book suggests while debugging:
# device = torch.device("cpu")

## 2. Load the tokenizer and model

This cell is the slow one (~1.4 GB of weights). Run it once; the later cells reuse `model`.

In [ ]:
QWEN3_DIR = Path("qwen3")
tokenizer_path = QWEN3_DIR / "tokenizer-base.json"
model_path = QWEN3_DIR / "qwen3-0.6B-base.pth"

tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path))
model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded Qwen3 0.6B base: {n_params:,} parameters on {device}")

## 3. A helper to generate text

Streams tokens as they are produced and prints timing via the book's
`generate_stats`. `use_cache=True` uses the KV-cache version from listing 2.4,
which avoids re-processing the whole sequence at every step.

In [ ]:
def generate(prompt, max_new_tokens=200, use_cache=True):
    """Stream a completion for `prompt` and report tokens/sec."""
    input_token_ids_tensor = torch.tensor(
        tokenizer.encode(prompt), device=device
    ).unsqueeze(0)

    stream = (
        generate_text_basic_stream_cache if use_cache
        else generate_text_basic_stream
    )

    print(f"{prompt}", end="", flush=True)
    generated_ids = []
    start_time = time.time()

    for token in stream(
        model=model,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
    ):
        print(tokenizer.decode(token.squeeze(0).tolist()), end="", flush=True)
        generated_ids.append(token.squeeze(0))

    end_time = time.time()
    print("\n")

    if generated_ids:
        generate_stats(
            torch.cat(generated_ids, dim=0), tokenizer, start_time, end_time
        )

## 4. Generate

The book's example prompt from section 2.7:

In [ ]:
generate("Explain large language models in a single sentence.")

Compare the two code paths — same output, very different speed:

In [ ]:
prompt = "The capital of France is"

print("--- without KV cache ---")
generate(prompt, max_new_tokens=40, use_cache=False)
print("\n--- with KV cache ---")
generate(prompt, max_new_tokens=40, use_cache=True)

## 5. Your turn

Edit the prompt and re-run. Things to keep in mind about a **base** model:

- It was only pretrained, so it continues text rather than reliably following
  instructions. Phrasing a prompt as a prefix to be continued often works
  better than asking a question.
- It will happily keep generating past the end of a good answer, and past
  `<|endoftext|>`, until it hits `max_new_tokens`.
- Generation is greedy (`argmax`), so the same prompt always gives the same
  output. There is no temperature or sampling yet. Greedy decoding also makes
  the model prone to repeat loops — the last cell below falls into one, which
  is expected here rather than a bug.

Teaching it to reason before answering is what the rest of the book is about.

In [ ]:
generate("Q: If a train travels 60 miles in 1.5 hours, what is its average speed?\nA:")

In [ ]:
generate(
    "A list of three facts about the Moon:\n1.",
    max_new_tokens=120,
)